In [1]:
import pandas as pd
import glob
import os

# Cesta ke složce s daty
data_folder = "Data"

# Pokud chceš otestovat úplně VŠECHNY CSV soubory ve složce najednou:
all_csv_files = [f for f in os.listdir(data_folder) if f.endswith('.csv')]

print(f"--- Zahájení kontroly datových souborů (Celkem nalezeno: {len(all_csv_files)}) ---\n")

summary = []

for file_name in all_csv_files:
    file_path = os.path.join(data_folder, file_name)
    try:
        # Zkusíme načíst soubor (předpokládáme středník a utf-8, co jsme řešili minule)
        # Pokud by to házelo chybu u chřipky z roku 1994, zkus změnit na encoding='cp1250'
        df = pd.read_csv(file_path, sep=';', encoding='utf-8')
        
        row_count = len(df)
        col_count = len(df.columns)
        first_col = df.columns[0]
        
        summary.append({
            "Soubor": file_name,
            "Stav": "✅ OK",
            "Řádků": row_count,
            "Sloupců": col_count,
            "První sloupec": first_col
        })
    except Exception as e:
        summary.append({
            "Soubor": file_name,
            "Stav": f"❌ CHYBA: {str(e)[:50]}...",
            "Řádků": 0,
            "Sloupců": 0,
            "První sloupec": "-"
        })

# Převedení výsledků do tabulky pro lepší přehlednost
report_df = pd.DataFrame(summary)
display(report_df)

print("\n--- Kontrola dokončena ---")

--- Zahájení kontroly datových souborů (Celkem nalezeno: 23) ---



,Soubor,Stav,Řádků,Sloupců,První sloupec
0,Hospitalizace_umrti_chripka(1994-2024).csv,✅ OK,17763,9,rok
1,Hospitalizace_umrti_Covid-19(2020-2025)_CR.csv,✅ OK,2136,124,Datum - ČR
2,Hospitalizace_umrti_Covid-19(2020-2025)_HKK.csv,✅ OK,1816,124,Datum - Královéhradecký kraj
3,Hospitalizace_umrti_Covid-19(2020-2025)_HMP.csv,✅ OK,2066,124,Datum - Hlavní město Praha
4,Hospitalizace_umrti_Covid-19(2020-2025)_JHC.csv,✅ OK,1794,124,Datum - Jihočeský kraj
5,Hospitalizace_umrti_Covid-19(2020-2025)_JMK.csv,✅ OK,2009,124,Datum - Jihomoravský kraj
6,Hospitalizace_umrti_Covid-19(2020-2025)_KVK.csv,✅ OK,1587,124,Datum - Karlovarský kraj
7,Hospitalizace_umrti_Covid-19(2020-2025)_LBK.csv,✅ OK,1660,124,Datum - Liberecký kraj
8,Hospitalizace_umrti_Covid-19(2020-2025)_MSK.csv,✅ OK,2040,124,Datum - Moravskoslezský kraj
9,Hospitalizace_umrti_Covid-19(2020-2025)_OLK.csv,✅ OK,1768,124,Datum - Olomoucký kraj



--- Kontrola dokončena ---


In [1]:
import pandas as pd
import glob
import os

data_folder = "Data"

# --- 1. POMOCNÉ FUNKCE A ČÍSELNÍKY PRO PŘEVOD OKRES -> KRAJ ---

# Pomocný slovník pro převod zkratek ze souborů na NUTS kód a celý název
mapovani_zkratek = {
    'HMP': ('CZ010', 'Hlavní město Praha'),
    'STC': ('CZ020', 'Středočeský kraj'),
    'JHC': ('CZ031', 'Jihočeský kraj'),
    'PLK': ('CZ032', 'Plzeňský kraj'),
    'KVK': ('CZ041', 'Karlovarský kraj'),
    'ULK': ('CZ042', 'Ústecký kraj'),
    'LBK': ('CZ051', 'Liberecký kraj'),
    'HKK': ('CZ052', 'Královéhradecký kraj'),
    'PAK': ('CZ053', 'Pardubický kraj'),
    'VYS': ('CZ063', 'Kraj Vysočina'),
    'JMK': ('CZ064', 'Jihomoravský kraj'),
    'OLK': ('CZ071', 'Olomoucký kraj'),
    'ZLK': ('CZ072', 'Zlínský kraj'),
    'MSK': ('CZ080', 'Moravskoslezský kraj')
}

# Číselník názvů okresů (pro očkování)
okres_to_kraj_full = {
    'Benešov': 'Středočeský kraj', 'Beroun': 'Středočeský kraj', 'Kladno': 'Středočeský kraj',
    'Kolín': 'Středočeský kraj', 'Kutná Hora': 'Středočeský kraj', 'Mělník': 'Středočeský kraj', 
    'Mladá Boleslav': 'Středočeský kraj', 'Nymburk': 'Středočeský kraj', 'Praha-východ': 'Středočeský kraj', 
    'Praha-západ': 'Středočeský kraj', 'Příbram': 'Středočeský kraj', 'Rakovník': 'Středočeský kraj',
    'České Budějovice': 'Jihočeský kraj', 'Český Krumlov': 'Jihočeský kraj', 'Jindřichův Hradec': 'Jihočeský kraj',
    'Písek': 'Jihočeský kraj', 'Prachatice': 'Jihočeský kraj', 'Strakonice': 'Jihočeský kraj', 
    'Tábor': 'Jihočeský kraj', 'Plzeň-město': 'Plzeňský kraj', 'Plzeň-jih': 'Plzeňský kraj', 
    'Plzeň-sever': 'Plzeňský kraj', 'Domažlice': 'Plzeňský kraj', 'Klatovy': 'Plzeňský kraj',
    'Rokycany': 'Plzeňský kraj', 'Tachov': 'Plzeňský kraj', 'Cheb': 'Karlovarský kraj',
    'Karlovy Vary': 'Karlovarský kraj', 'Sokolov': 'Karlovarský kraj', 'Děčín': 'Ústecký kraj', 
    'Chomutov': 'Ústecký kraj', 'Litoměřice': 'Ústecký kraj', 'Louny': 'Ústecký kraj',
    'Most': 'Ústecký kraj', 'Teplice': 'Ústecký kraj', 'Ústí nad Labem': 'Ústecký kraj',
    'Česká Lípa': 'Liberecký kraj', 'Jablonec nad Nisou': 'Liberecký kraj', 'Liberec': 'Liberecký kraj',
    'Semily': 'Liberecký kraj', 'Hradec Králové': 'Královéhradecký kraj', 'Jičín': 'Královéhradecký kraj',
    'Náchod': 'Královéhradecký kraj', 'Rychnov nad Kněžnou': 'Královéhradecký kraj',
    'Trutnov': 'Královéhradecký kraj', 'Chrudim': 'Pardubický kraj', 'Pardubice': 'Pardubický kraj',
    'Svitavy': 'Pardubický kraj', 'Ústí nad Orlicí': 'Pardubický kraj', 'Havlíčkův Brod': 'Kraj Vysočina', 
    'Jihlava': 'Kraj Vysočina', 'Pelhřimov': 'Kraj Vysočina', 'Třebíč': 'Kraj Vysočina',
    'Žďár nad Sázavou': 'Kraj Vysočina', 'Blansko': 'Jihomoravský kraj', 'Brno-město': 'Jihomoravský kraj', 
    'Brno-venkov': 'Jihomoravský kraj', 'Břeclav': 'Jihomoravský kraj', 'Hodonín': 'Jihomoravský kraj',
    'Vyškov': 'Jihomoravský kraj', 'Znojmo': 'Jihomoravský kraj', 'Jeseník': 'Olomoucký kraj',
    'Olomouc': 'Olomoucký kraj', 'Prostějov': 'Olomoucký kraj', 'Přerov': 'Olomoucký kraj',
    'Šumperk': 'Olomoucký kraj', 'Kroměříž': 'Zlínský kraj', 'Uherské Hradiště': 'Zlínský kraj',
    'Vsetín': 'Zlínský kraj', 'Zlín': 'Zlínský kraj', 'Bruntál': 'Moravskoslezský kraj',
    'Frýdek-Místek': 'Moravskoslezský kraj', 'Karviná': 'Moravskoslezský kraj', 'Nový Jičín': 'Moravskoslezský kraj',
    'Opava': 'Moravskoslezský kraj', 'Ostrava-město': 'Moravskoslezský kraj', 'Hl. m. Praha': 'Hlavní město Praha'
}

# Číselník kódů okresů (pro detailní úmrtí)
kod_kraje_to_nazev = {
    'CZ010': 'Hlavní město Praha', 'CZ020': 'Středočeský kraj', 'CZ031': 'Jihočeský kraj',
    'CZ032': 'Plzeňský kraj', 'CZ041': 'Karlovarský kraj', 'CZ042': 'Ústecký kraj',
    'CZ051': 'Liberecký kraj', 'CZ052': 'Královéhradecký kraj', 'CZ053': 'Pardubický kraj',
    'CZ063': 'Kraj Vysočina', 'CZ064': 'Jihomoravský kraj', 'CZ071': 'Olomoucký kraj',
    'CZ072': 'Zlínský kraj', 'CZ080': 'Moravskoslezský kraj'
}

nazev_to_kod = {v: k for k, v in kod_kraje_to_nazev.items()}

def agreguj_chripku_na_kraje(df):
    # 1. Přiřazení celého názvu kraje podle okresu (používá tvůj okres_to_kraj_full)
    df['Kraj_Název'] = df['uzemni_celek'].map(okres_to_kraj_full)
    
    # 2. Vyčištění od nenamapovaných řádků
    df_clean = df.dropna(subset=['Kraj_Název']).copy()
    
    # 3. Agregace dat (sumarizace za kraj a sezónu)
    df_agg = df_clean.groupby(['Kraj_Název', 'sezona']).agg({
        'pocet_vakcinovanych': 'sum',
        'demografie': 'sum'
    }).reset_index()
    
    # 4. DOPLNĚNÍ Kraj_ID s využitím tvého číselníku kod_kraje_to_nazev
    # Prohodíme klíče a hodnoty: {'Název': 'CZxxx'}
    df_agg.insert(0, 'Kraj_ID', df_agg['Kraj_Název'].map(nazev_to_kod))
    
    # 5. Výpočet proočkovanosti v %
    df_agg['proockovanost_procenta'] = (df_agg['pocet_vakcinovanych'] / df_agg['demografie']) * 100
    
    return df_agg

def preved_detailni_umrti_na_kraje(df):
    df_new = df.copy()
    # Extrakce kódu kraje z kódu okresu (CZ0531 -> CZ053)
    df_new['Kod_Kraje'] = df_new['okres_bydliste'].str[:5]
    df_new['Kraj_Název'] = df_new['Kod_Kraje'].map(kod_kraje_to_nazev)
    return df_new

def dopln_id_podle_nazvu(df):
    if 'Kraj_ID' not in df.columns:
        df.insert(0, 'Kraj_ID', df['Kraj'].map(nazev_to_kod))
    return df

# --- 2. SLOUČENÁ KRAJSKÁ DATA COVID-19 ---
kraj_files = glob.glob(os.path.join(data_folder, "Hospitalizace_umrti_Covid-19(2020-2025)_*.csv"))
kraj_files = [f for f in kraj_files if "_CR.csv" not in f]

seznam_df = []

for file in kraj_files:
    temp_df = pd.read_csv(file, sep=';', encoding='utf-8')
    
    # Sjednocení prvního sloupce na 'Datum'
    temp_df.rename(columns={temp_df.columns[0]: 'Datum'}, inplace=True)
    
    # Získání zkratky z názvu souboru (např. 'HKK')
    zkratka = os.path.basename(file).split('_')[-1].replace('.csv', '')
    
    # Získání NUTS kódu a celého názvu ze slovníku
    nuts_kod, plny_nazev = mapovani_zkratek.get(zkratka, (zkratka, "Neznámý kraj"))
    
    # Vložení nových identifikačních sloupců na začátek tabulky
    temp_df.insert(0, 'Kraj_Název', plny_nazev)
    temp_df.insert(0, 'Kraj_ID', nuts_kod) # NUTS kód (CZxxx)
    
    seznam_df.append(temp_df)

df_covid_umrti_hosp_kraje = pd.concat(seznam_df, ignore_index=True)
df_covid_umrti_hosp_kraje['Datum'] = pd.to_datetime(df_covid_umrti_hosp_kraje['Datum'], dayfirst=True)

# --- 3. COVID-19: CELOREPUBLIKOVÁ DATA ---
df_covid_umrti_hosp_cr = pd.read_csv(os.path.join(data_folder, "Hospitalizace_umrti_Covid-19(2020-2025)_CR.csv"), sep=';', encoding='utf-8')
df_covid_umrti_hosp_cr.rename(columns={df_covid_umrti_hosp_cr.columns[0]: 'Datum'}, inplace=True)
df_covid_umrti_hosp_cr['Datum'] = pd.to_datetime(df_covid_umrti_hosp_cr['Datum'], dayfirst=True)

# --- 4. OČKOVÁNÍ: COVID-19 ---
df_covid_ockov_kraje = pd.read_csv(os.path.join(data_folder, "Ockovani_kraj_Covid-19(2020-2025).csv"), sep=';', encoding='utf-8')

# --- 5. CHŘIPKA: HOSPITALIZACE A ÚMRTÍ (HISTORIE) ---
df_chripka_umrti_hosp_kraje = pd.read_csv(os.path.join(data_folder, "Hospitalizace_umrti_chripka(1994-2024).csv"), sep=';', encoding='utf-8')

# --- 6. CHŘIPKA: ÚMRTÍ (DETAILNÍ DATA 2015-2024 + TRANSFORMACE) ---
df_raw_umrti_detail = pd.read_csv(os.path.join(data_folder, "Umrti_chripka(2015-2024).csv"), sep=';', encoding='utf-8')
df_chripka_umrti_kraje = preved_detailni_umrti_na_kraje(df_raw_umrti_detail)

# --- 7. CHŘIPKA: KRAJSKÉ SROVNÁNÍ ÚMRTÍ (ABS, NA 100tis, PROCENTA) ---
df_chripka_umrti_abs_kraje = pd.read_csv(os.path.join(data_folder, "Umrti_chripka_kraje_abs(1994-2023).csv"), sep=';', encoding='utf-8')
df_chripka_umrti_norm_kraje = pd.read_csv(os.path.join(data_folder, "Umrti_chripka_kraje_na100tis(1994-2023).csv"), sep=';', encoding='utf-8')
df_chripka_umrti_perc_kraje = pd.read_csv(os.path.join(data_folder, "Umrti_chripka_kraje_procenta(1994-2023).csv"), sep=';', encoding='utf-8')

df_chripka_umrti_abs_kraje = dopln_id_podle_nazvu(df_chripka_umrti_abs_kraje)
df_chripka_umrti_norm_kraje = dopln_id_podle_nazvu(df_chripka_umrti_norm_kraje)
df_chripka_umrti_perc_kraje = dopln_id_podle_nazvu(df_chripka_umrti_perc_kraje)

# --- 8. OČKOVÁNÍ: CHŘIPKA (Transformace z okresů na celé názvy krajů) ---
df_raw_65 = pd.read_csv(os.path.join(data_folder, "Ockovani_chripka(2010-2025)_65plus.csv"), sep=';', encoding='utf-8')
df_raw_vsichni = pd.read_csv(os.path.join(data_folder, "Ockovani_chripka(2010-2025)_vsichni.csv"), sep=';', encoding='utf-8')

df_chripka_ockov_65plus_kraje = agreguj_chripku_na_kraje(df_raw_65)
df_chripka_ockov_vsichni_kraje = agreguj_chripku_na_kraje(df_raw_vsichni)

print("✅ Všech 10 datových celků bylo úspěšně načteno a sjednoceno na úroveň krajů.")

✅ Všech 10 datových celků bylo úspěšně načteno a sjednoceno na úroveň krajů.


In [2]:
# --- KONTROLNÍ BLOK IMPORTU ---

# Seznam proměnných a jejich popisů pro report
datasets = {
    "df_covid_umrti_hosp_kraje": "COVID-19 - Sloučené kraje (14 souborů)",
    "df_covid_umrti_hosp_cr": "COVID-19 - Celá ČR",
    "df_covid_ockov_kraje": "Očkování COVID-19 - Krajské",
    "df_chripka_umrti_hosp_kraje": "Chřipka - Historické hosp. (1994-2024)",
    "df_chripka_umrti_kraje": "Chřipka - Detailní úmrtí (2015-2024)",
    "df_chripka_umrti_abs_kraje": "Chřipka - Úmrtí kraje (Absolutně)",
    "df_chripka_umrti_norm_kraje": "Chřipka - Úmrtí kraje (na 100 tis.)",
    "df_chripka_umrti_perc_kraje": "Chřipka - Úmrtí kraje (Procenta)",
    "df_chripka_ockov_65plus_kraje": "Očkování chřipka - 65+",
    "df_chripka_ockov_vsichni_kraje": "Očkování chřipka - Všichni"
}

check_results = []

for var_name, description in datasets.items():
    try:
        # Získáme objekt proměnné podle jejího názvu
        obj = globals().get(var_name)
        
        if obj is not None:
            check_results.append({
                "Proměnná": var_name,
                "Popis dat": description,
                "Stav": "✅ Načteno",
                "Počet řádků": len(obj),
                "Počet sloupců": len(obj.columns)
            })
        else:
            check_results.append({
                "Proměnná": var_name,
                "Popis dat": description,
                "Stav": "❌ Chybí (nenalezeno)",
                "Počet řádků": 0,
                "Počet sloupců": 0
            })
    except Exception as e:
        check_results.append({
            "Proměnná": var_name,
            "Popis dat": description,
            "Stav": f"⚠️ Chyba: {str(e)[:30]}",
            "Počet řádků": 0,
            "Počet sloupců": 0
        })

# Zobrazení výsledného reportu
final_report = pd.DataFrame(check_results)
print("--- FINÁLNÍ KONTROLA DATOVÝCH SAD V PAMĚTI ---")
display(final_report)

# Malý test kontinuity datumu u hlavního datasetu
if 'df_covid_umrti_hosp_kraje' in globals():
    min_date = df_covid_umrti_hosp_kraje['Datum'].min().strftime('%d.%m.%Y')
    max_date = df_covid_umrti_hosp_kraje['Datum'].max().strftime('%d.%m.%Y')
    print(f"\n📅 Časový rozsah COVID-19 (kraje): {min_date} až {max_date}")

--- FINÁLNÍ KONTROLA DATOVÝCH SAD V PAMĚTI ---


,Proměnná,Popis dat,Stav,Počet řádků,Počet sloupců
0,df_covid_umrti_hosp_kraje,COVID-19 - Sloučené kraje (14 souborů),✅ Načteno,25402,126
1,df_covid_umrti_hosp_cr,COVID-19 - Celá ČR,✅ Načteno,2136,124
2,df_covid_ockov_kraje,Očkování COVID-19 - Krajské,✅ Načteno,357518,9
3,df_chripka_umrti_hosp_kraje,Chřipka - Historické hosp. (1994-2024),✅ Načteno,17763,9
4,df_chripka_umrti_kraje,Chřipka - Detailní úmrtí (2015-2024),✅ Načteno,3295,11
5,df_chripka_umrti_abs_kraje,Chřipka - Úmrtí kraje (Absolutně),✅ Načteno,103,33
6,df_chripka_umrti_norm_kraje,Chřipka - Úmrtí kraje (na 100 tis.),✅ Načteno,103,33
7,df_chripka_umrti_perc_kraje,Chřipka - Úmrtí kraje (Procenta),✅ Načteno,103,33
8,df_chripka_ockov_65plus_kraje,Očkování chřipka - 65+,✅ Načteno,210,6
9,df_chripka_ockov_vsichni_kraje,Očkování chřipka - Všichni,✅ Načteno,210,6



📅 Časový rozsah COVID-19 (kraje): 11.03.2020 až 13.01.2026


In [ ]:
# Seznam vašich finálních datasetů pro kontrolu
datasets = {
    "COVID: Hospitalizace a úmrtí (kraje)": df_covid_umrti_hosp_kraje,
    "COVID: Očkování (kraje)": df_covid_ockov_kraje,
    "Chřipka: Hospitalizace a úmrtí (historie)": df_chripka_umrti_hosp_kraje,
    "Chřipka: Úmrtí (detailní)": df_chripka_umrti_kraje,
    "Chřipka: Úmrtí ABS (historie)": df_chripka_umrti_abs_kraje,
    "Chřipka: Úmrtí na 100tis": df_chripka_umrti_norm_kraje,
    "Chřipka: Úmrtí %": df_chripka_umrti_perc_kraje,
    "Chřipka: Očkování 65+": df_chripka_ockov_65plus_kraje,
    "Chřipka: Očkování všichni": df_chripka_ockov_vsichni_kraje,
    "COVID: Celorepubliková data": df_covid_umrti_hosp_cr
}

for nazev, df in datasets.items():
    print(f"\n--- {nazev} ---")
    # Zobrazíme jen první 2 řádky pro přehlednost
    display(df.iloc[200:202])


--- COVID: Hospitalizace a úmrtí (kraje) ---


,Kraj_ID,Kraj_Název,Datum,Celkový počet nakažených,Počet hospitalizovaných celkem v daném dni,Počet hospitalizovaných na JIP celkem v daném dni,Počet nově hospitalizovaných celkem,Počet nově hospitalizovaných na JIP,Počet zemřelých,Počet antigenních testů,...,Zemřelí ve věku 45-49 let,Zemřelí ve věku 50-54 let,Zemřelí ve věku 55-59 let,Zemřelí ve věku 60-64 let,Zemřelí ve věku 65-69 let,Zemřelí ve věku 70-74 let,Zemřelí ve věku 75-79 let,Zemřelí ve věku 80-84 let,Zemřelí ve věku 85+ let,Zemřelí - věk neznámý
200,CZ052,Královéhradecký kraj,2020-11-05,719.0,458.0,88.0,49.0,15.0,16.0,115.0,...,0.0,0.0,0.0,0.0,1.0,1.0,6.0,2.0,6.0,0.0
201,CZ052,Královéhradecký kraj,2020-11-06,596.0,450.0,89.0,55.0,18.0,11.0,82.0,...,0.0,0.0,0.0,0.0,1.0,4.0,1.0,1.0,4.0,0.0



--- COVID: Očkování (kraje) ---


,id,datum,vakcina,kraj_nuts_kod,kraj_nazev,vekova_skupina,prvnich_davek,druhych_davek,celkem_davek
200,49fc9d1a-aa2d-4cff-ac1a-0313608b07eb,31.12.2020,Comirnaty,CZ064,Jihomoravský kraj,16-17,1,0,1
201,ceeb4b4c-e453-4186-8da1-3ff0e29e11f2,31.12.2020,Comirnaty,CZ064,Jihomoravský kraj,40-44,71,0,71



--- Chřipka: Hospitalizace a úmrtí (historie) ---


...

,Kraj_ID,Kraj_Název,sezona,pocet_vakcinovanych,demografie,proockovanost_procenta
200,CZ042,Ústecký kraj,2015-2016,30238,145466.0,20.786988
201,CZ042,Ústecký kraj,2016-2017,33361,150338.0,22.190664



--- Chřipka: Očkování všichni ---


,Kraj_ID,Kraj_Název,sezona,pocet_vakcinovanych,demografie,proockovanost_procenta
200,CZ042,Ústecký kraj,2015-2016,38505,822826.0,4.679604
201,CZ042,Ústecký kraj,2016-2017,41896,821377.0,5.100703



--- COVID: Celorepubliková data ---


,Datum,Celkový počet nakažených,Počet hospitalizovaných celkem v daném dni,Počet hospitalizovaných na JIP celkem v daném dni,Počet nově hospitalizovaných celkem,Počet nově hospitalizovaných na JIP,Počet zemřelých,Počet antigenních testů,Počet PCR testů,počet testů CELKEM,...,Zemřelí ve věku 45-49 let,Zemřelí ve věku 50-54 let,Zemřelí ve věku 55-59 let,Zemřelí ve věku 60-64 let,Zemřelí ve věku 65-69 let,Zemřelí ve věku 70-74 let,Zemřelí ve věku 75-79 let,Zemřelí ve věku 80-84 let,Zemřelí ve věku 85+ let,Zemřelí - věk neznámý
200,2020-09-27,1308.0,823.0,178.0,119.0,28.0,16.0,0.0,10468.0,10468.0,...,0.0,0.0,1.0,1.0,0.0,3.0,1.0,2.0,8.0,0.0
201,2020-09-28,1286.0,845.0,191.0,102.0,43.0,11.0,0.0,12218.0,12218.0,...,1.0,0.0,0.0,0.0,0.0,2.0,3.0,1.0,4.0,0.0
